# Borrower / Investor Classification Pipeline

This notebook runs the full pipeline: data loading, cleaning, pseudo-label generation, EDA, model training, evaluation, explainability, and artifact creation.

## Roadmap

1. Load the workbook and inspect the `REAL DATA` sheet.
2. Clean raw values and engineer credit/investment features.
3. Generate pseudo-labels when verified labels are missing.
4. Train baseline models and a final production model.
5. Evaluate with holdout metrics, cross-validation, confusion matrix, ROC-AUC, PR-AUC, calibration, and feature importance.
6. Save model and inference artifacts.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('../data/raw/Machine_Learning_Project(Sample).xlsx')
SHEET = 'REAL DATA'

# Change DATA_PATH if your workbook is stored elsewhere.
raw = pd.read_excel(DATA_PATH, sheet_name=SHEET)
raw.head()

In [ ]:
raw.shape, raw.columns.tolist()[:10], raw.columns.tolist()[-10:]

In [ ]:
from src.data_processing import clean_and_engineer, generate_pseudo_labels, make_model_frame, split_feature_types

cleaned = clean_and_engineer(raw)
labeled = generate_pseudo_labels(cleaned)
labeled['segment_label'].value_counts()

In [ ]:
X = make_model_frame(labeled)
y = labeled['segment_label']
num, cat = split_feature_types(X)
print('Rows:', X.shape[0])
print('Features:', X.shape[1])
print('Numeric:', len(num))
print('Categorical:', len(cat))

In [ ]:
# Train from the notebook by calling the same production script.
# Use --fast for quick iteration. Remove --fast for fuller CV/tuning.
!python ../train.py --data ../data/raw/Machine_Learning_Project\(Sample\).xlsx --sheet 'REAL DATA' --fast

In [ ]:
import json
metrics = json.load(open('../reports/metrics.json'))
metrics['test_macro_f1'], metrics['test_micro_f1'], metrics.get('final_model_cv_macro_f1_mean')

In [ ]:
print(open('../reports/classification_report.txt').read())

In [ ]:
from IPython.display import Image, display
for fig in ['class_balance.png', 'confusion_matrix.png', 'global_feature_importance.png', 'shap_global_importance.png', 'calibration_curve.png']:
    path = Path('../reports/figures') / fig
    if path.exists():
        display(Image(filename=str(path)))

In [ ]:
# Example inference
!python ../inference.py --input ../artifacts/sample_request.json --model ../artifacts/model.joblib

## Production note

The current model learns the pseudo-labeling framework. Before using the model for automated decisions, replace pseudo-labels with verified labels or validate the rules with domain experts.